# Class 4 · a rice leaf doctor (CNN)
Instructor: Nazmus Sayadat Rumman. The last cell saves **class4_ship.zip** with every file the app needs for Class 4.

**Before Run All (Add Input):** 1) the RiceLeafBD photos, 2) `rice_split.csv`.
**Settings → Accelerator → GPU.** Run All takes about 6 minutes on GPU.

**D4-01 · Setup**

In [ ]:
import glob, json, os, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras
keras.utils.set_random_seed(42)
CLASSES = ["Blast", "Narrow_Brown_Spot", "Normal_Leaf", "Sheath_Blight", "Tungro"]
hits = glob.glob("/kaggle/input/**/Tungro", recursive=True) + glob.glob("**/Tungro", recursive=True)
assert hits, "RiceLeafBD photos not found. Right panel -> Add Input -> RiceLeafBD dataset, then run this cell again."
ROOT = os.path.dirname(hits[0])
print("TensorFlow", tf.__version__, "| GPU:", len(tf.config.list_physical_devices("GPU")), "| photos in", ROOT)

**D4-02 · Meet the five classes** (cover the titles first and let the class guess)

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(16, 3.6))
for a, c in zip(ax, CLASSES):
    files = sorted(glob.glob(f"{ROOT}/{c}/*.jpg"))
    a.imshow(plt.imread(files[0])); a.set_title(f"{c}\n{len(files)} photos"); a.axis("off")
plt.show()

**D4-03 · A photo is just numbers**

In [ ]:
photo = sorted(glob.glob(f"{ROOT}/Blast/*.jpg"))[0]
img = tf.io.decode_image(tf.io.read_file(photo), channels=3)
print("shape:", img.shape, "| one pixel (R, G, B):", img[150, 150].numpy())
leaf = tf.image.resize(img, [160, 160])          # every photo becomes 160 x 160
print("after resize:", leaf.shape, "=", 160 * 160 * 3, "numbers per photo")

**D4-04 · Train / validation / test** (the split was made once, so near-identical photos never sit on both sides)

In [ ]:
split = pd.read_csv((glob.glob("/kaggle/input/**/rice_split.csv", recursive=True) + ["data/rice_split.csv"])[0])
def load_image(path):                              # the API uses this exact function
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, [160, 160])
def make_ds(name):                                 # photos of one part -> batches of 32
    df = split[split["split"] == name]
    paths, labels = list(ROOT + "/" + df["cls"] + "/" + df["file"]), df["cls"].map(CLASSES.index).to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels)).map(lambda p, y: (load_image(p), y)).cache()
    return (ds.shuffle(3000) if name == "train" else ds).batch(32), paths, labels
(train_ds, _, _), (val_ds, _, _), (test_ds, test_paths, y_img) = [make_ds(s) for s in ["train", "val", "test"]]
print(split["split"].value_counts().to_dict())       # a few photos are marked "excluded" and not used

**D4-05 · One 3×3 filter by hand: this is all a CNN does**

In [ ]:
edge = tf.constant([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], tf.float32)   # "is this pixel different from its neighbours?"
gray = tf.image.rgb_to_grayscale(leaf)[None]                                # 1 x 160 x 160 x 1
out = tf.nn.relu(tf.nn.conv2d(gray, tf.reshape(edge, [3, 3, 1, 1]), strides=1, padding="SAME"))
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(gray[0, ..., 0], cmap="gray"); ax[0].set_title("the leaf"); ax[0].axis("off")
ax[1].imshow(out[0, ..., 0], cmap="gray"); ax[1].set_title("after one 3x3 filter"); ax[1].axis("off")
plt.show()

**D4-06 · Build the CNN: four blocks of [filters → normalise → ReLU → shrink]**

In [ ]:
def block(n):    # n filters of size 3x3, then halve the picture
    return [keras.layers.Conv2D(n, 3, padding="same"), keras.layers.BatchNormalization(),
            keras.layers.ReLU(name=f"relu_{n}"), keras.layers.MaxPooling2D()]
cnn = keras.Sequential([keras.Input((160, 160, 3)), keras.layers.Rescaling(1 / 255),
                          keras.layers.RandomFlip("horizontal_and_vertical"), keras.layers.RandomRotation(0.1),   # new photo every epoch
                          *block(16), *block(32), *block(64), *block(128),
                          keras.layers.GlobalAveragePooling2D(), keras.layers.Dropout(0.3),
                          keras.layers.Dense(5, activation="softmax")])
print("parameters:", f"{cnn.count_params():,}")

**D4-07 · Train, and stop when validation stops improving** (pre-run: about 5 minutes on GPU)

In [ ]:
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
stop = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
slow = keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=3)   # smaller steps when stuck
hist = cnn.fit(train_ds, validation_data=val_ds, epochs=60, callbacks=[stop, slow], verbose=2)
best = int(np.argmin(hist.history["val_loss"]))
print("ran", len(hist.history["loss"]), "epochs | kept epoch", best + 1,
      "| validation accuracy", round(hist.history["val_accuracy"][best], 4))

**D4-08 · The learning curve**

In [ ]:
plt.plot(hist.history["accuracy"], label="train"); plt.plot(hist.history["val_accuracy"], label="validation")
plt.axvline(best, ls="--", c="gray"); plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.show()

**D4-09 · The test set, used once**

In [ ]:
probs = cnn.predict(test_ds, verbose=0); pred = probs.argmax(1); names = np.array(CLASSES)
print("TEST accuracy:", round((pred == y_img).mean(), 4), "| guessing at random would give 0.2")
print(pd.crosstab(names[y_img], names[pred], rownames=["true"], colnames=["CNN said"]))

**D4-10 · The CNN's report card on 10 test photos** (green = right, red = wrong)

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(16, 7))
for a, i in zip(ax.flat, np.random.default_rng(7).choice(len(y_img), 10, replace=False)):
    a.imshow(plt.imread(test_paths[i])); a.axis("off")
    a.set_title(f"true: {names[y_img[i]]}\nCNN: {names[pred[i]]} {probs[i].max():.0%}",
                color="green" if pred[i] == y_img[i] else "red")
plt.show()

**D4-11 · What the CNN sees: the first 8 filters, then 8 filters three blocks deeper**

In [ ]:
for layer in ["relu_16", "relu_64"]:
    maps = keras.Model(cnn.inputs, cnn.get_layer(layer).output)(leaf[None])[0]
    fig, ax = plt.subplots(1, 8, figsize=(16, 2.4)); fig.suptitle(f"{layer}: picture size {maps.shape[0]} x {maps.shape[1]}")
    for k, a in enumerate(ax): a.imshow(maps[..., k], cmap="viridis"); a.axis("off")
    plt.show()

**D4-12 · Save and zip: the CNN, its class names, 15 test photos for the game → class4_ship.zip**

In [ ]:
os.makedirs("ship/models", exist_ok=True); os.makedirs("ship/data/samples", exist_ok=True)
cnn.save("ship/models/rice_cnn.keras")
json.dump({"classes": CLASSES, "image_size": [160, 160]}, open("ship/models/rice_meta.json", "w"), indent=1)
for i in [i for c in range(5) for i in np.where(y_img == c)[0][:3]]:      # 3 test photos per class
    shutil.copy(test_paths[i], f"ship/data/samples/{CLASSES[y_img[i]]}__{os.path.basename(test_paths[i])}")
    print(f"{CLASSES[y_img[i]]:18s} CNN: {names[pred[i]]:18s} {probs[i].max():.1%}")   # the app must show the same
shutil.make_archive("class4_ship", "zip", "ship"); print("saved class4_ship.zip:", sorted(os.listdir("ship/models")))